<a href="https://colab.research.google.com/github/VatsalBarai/ExpenseTracker/blob/main/Vatsal_Barai_agents_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys

print("Attempting a more aggressive installation to resolve persistent import issues.")

print("Uninstalling existing langchain and related packages to ensure a clean slate...")
# Use -y -q for quiet and automatic 'yes' to prompts
!pip uninstall -y -q langchain langchain-core langchain-community langchain-groq langgraph langsmith ddgs

print("Performing a clean reinstallation of langchain and related packages...")
!pip install -q -U langchain langchain-community langchain-groq langgraph langsmith ddgs

import langchain
print(f"Successfully installed langchain version: {langchain.__version__}")

print("\n--- CRITICAL: IMMEDIATE ACTION REQUIRED ---")
print("To ensure all packages are correctly loaded, you MUST perform a **FULL COLAB RUNTIME RESTART** (Runtime > Restart runtime).")
print("After restarting, please **RUN ALL CELLS FROM THE BEGINNING** to properly initialize your environment.")
print("This step is absolutely essential for the new installations to take effect.")
print("------------------------------------------")


Attempting a more aggressive installation to resolve persistent import issues.
Uninstalling existing langchain and related packages to ensure a clean slate...
Performing a clean reinstallation of langchain and related packages...

--- CRITICAL: IMMEDIATE ACTION REQUIRED ---
To ensure all packages are correctly loaded, you MUST perform a **FULL COLAB RUNTIME RESTART** (Runtime > Restart runtime).
After restarting, please **RUN ALL CELLS FROM THE BEGINNING** to properly initialize your environment.
This step is absolutely essential for the new installations to take effect.
------------------------------------------


In [2]:
import os
from google.colab import userdata

# Access your API keys from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
WEATHER_API_KEY = userdata.get('WEATHER_API_KEY')

In [3]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests
from langsmith.client import Client as LangSmithClient

In [22]:
from langchain_core.tools import tool
from ddgs import DDGS

@tool
def duckduckgo_search(query: str) -> str:
    """Searches DuckDuckGo for the given query and returns the summary of the top result."""
    with DDGS() as ddgs_client:
        # ddgs_client.text returns a list of dictionaries. We take the 'body' of the first result.
        results = list(ddgs_client.text(query, max_results=1))
        if results and results[0].get('body'):
            return results[0]['body']
        return "No detailed search results found for the query."

search_tool = duckduckgo_search

In [23]:
from langchain_core.tools import tool
import requests

@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  # Access your Weather API key from Colab Secrets
  WEATHER_API_KEY = userdata.get('WEATHER_API_KEY')
  url = f'https://api.weatherstack.com/current?access_key={WEATHER_API_KEY}&query={city}'

  response = requests.get(url)

  return response.json()

In [24]:

llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile") # Changed to a currently supported Groq model

In [25]:
from langchain.agents import create_agent

In [26]:
agent = create_agent(
    model=llm,
    tools=[duckduckgo_search, get_weather_data],
    system_prompt="""
You are a helpful AI assistant.

Whenever needed, use the available tools.

Answer accurately.
"""
)

In [31]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the best place to visit in Rajkot in the early morning, then find its sunrise timings."
            }
        ]
    }
)

In [32]:
print(response['messages'][-1].content)

The best place to visit in Rajkot in the early morning is the Watson Museum. The sunrise timing in Rajkot is 06:06 AM.
